# Walkthrough: Dan's conformal solver, stage by stage

Companion to [`03_dans_method.md`](03_dans_method.md). Each section below runs one stage of
`mom6_forge/_conformal.py` (PR #151, head `db0db39`) through its **private** functions and draws
what that stage produced, first on a simple trapezoid and then on the California test outline.
The text here stays short; the "why" for each stage is in the section of `03_dans_method.md`
named in each heading.

Every plot cell saves a PNG to `learn/figures/pipeline_<name>.png`; `03_dans_method.md` embeds them.

The pipeline, as in `solve()`:

```
_validate_and_orient → as_projection → to_xy → order_corners           (§1–2)
_sides → _mesh → _stiffness → _harmonic(ξ) , _harmonic(η)              (§3–6)
aspect = √(E_ξ/E_η) → nx, ny                                           (§7)
_sample (inverse map) → snap sides → pin corners → to_lonlat            (§8–9)
```

## Setup

Imports the worktree's `mom6_forge`, not an installed copy (the assert checks that).
`FIG` is an absolute path, so the notebook writes its figures to `learn/figures/` wherever it runs.

In [ ]:
import os, sys, json, time
from pathlib import Path

WORKTREE = Path("/glade/u/home/manishrv/documents/croc/regional_mom_workflows/mom6_forge-worktrees/conformal-solver")
LEARN = Path("/glade/u/home/manishrv/documents/croc/dev/mom6_forge_gridsketcher/learn")
FIG = LEARN / "figures"
FIG.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("ESMFMKFILE", "/glade/work/manishrv/conda-envs/dev_mf/lib/esmf.mk")
sys.path.insert(0, str(WORKTREE))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.path import Path as MplPath
from matplotlib.tri import Triangulation
import shapely
from shapely.geometry import LinearRing, Polygon

import mom6_forge
import mom6_forge._conformal as cf

assert str(WORKTREE) in mom6_forge.__file__, mom6_forge.__file__
print(mom6_forge.__file__)

# Categorical slots (sides S, E, N, W; lattice vs band) and ink colours
SIDE = {"S": "#2a78d6", "E": "#eb6834", "N": "#1baf7a", "W": "#eda100"}
LATTICE, BAND, ALERT = "#2a78d6", "#eb6834", "#e34948"
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 130, "font.size": 10,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 11, "axes.titleweight": "bold",
})


def save(fig, name):
    path = FIG / f"pipeline_{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("saved", path)

### Helpers

`run_stages` calls the private stages exactly as `solve()` does and keeps every intermediate.
`_solve_mesh` doesn't return the two Dirichlet energies, so they are recomputed with `_harmonic`.
`lattice_classes` repeats `_mesh`'s lattice bookkeeping (`_conformal.py` L399–408) so the plots can
tell the structured squares from the Delaunay band; `_mesh` appends the squares last
(`np.vstack([tri[keep], squares])`, L423), so the last `2 × n_full` triangles are the squares.

In [ ]:
def lattice_classes(P, h):
    # Same lines as _mesh (L399-408): lattice coords, distance to the outline in units of h,
    # and the 'full' squares whose 4 corners are all more than 1.5 h from the outline.
    lo, ring, path = P.min(axis=0) + h / 2, LinearRing(P), MplPath(P)
    X, Y = np.meshgrid(*(np.arange(a, b, h) for a, b in zip(lo, P.max(axis=0))))
    xy, d = np.column_stack([X.ravel(), Y.ravel()]), np.zeros(X.size)
    ins = path.contains_points(xy)
    d[ins] = shapely.distance(shapely.points(xy[ins]), ring) / h
    d = d.reshape(X.shape)
    deep = d > 1.5
    full = deep[:-1, :-1] & deep[:-1, 1:] & deep[1:, :-1] & deep[1:, 1:]
    return X, Y, d, full


def run_stages(lon, lat, corners, n_cells, projection, raster_points, grid=True):
    t0 = time.perf_counter()
    f = cf._solve_mesh(lon, lat, corners, n_cells, projection, raster_points)
    K = cf._stiffness(f["nodes"], f["tri"])
    _, f["E_xi"] = cf._harmonic(K, f["ids"][3], f["ids"][1])
    _, f["E_eta"] = cf._harmonic(K, f["ids"][0], f["ids"][2])
    f["K"] = K
    f["X"], f["Y"], f["d"], full = lattice_classes(f["P"], f["h"])
    f["n_sq"] = 2 * int(full.sum())
    if grid:
        f["cg"] = cf.solve(lon, lat, corners, n_cells, projection, raster_points)
    print(f"{f['projection'].kind}: raster_points={raster_points:,}  h={f['h']/1e3:.3f} km  "
          f"nodes={len(f['nodes']):,}  triangles={len(f['tri']):,} "
          f"(Delaunay band {len(f['tri']) - f['n_sq']:,}, lattice squares {f['n_sq']:,})  "
          f"[{time.perf_counter() - t0:.2f} s]")
    print(f"  corners {f['corners']}  angles {np.round(f['corner_angles'], 1).tolist()}")
    print(f"  E_xi={f['E_xi']:.4f}  E_eta={f['E_eta']:.4f}  E_xi*E_eta={f['E_xi']*f['E_eta']:.4f}  "
          f"aspect={f['aspect']:.4f}  nx, ny = {f['nx']}, {f['ny']}")
    return f


def draw_outline(ax, f, lw=2.2, labels=True, s=1e3, corner_labels=True):
    for k, (pts, _) in enumerate(f["sides"]):
        c = SIDE["SENW"[k]]
        ax.plot(pts[:, 0] / s, pts[:, 1] / s, color=c, lw=lw, solid_capstyle="round", zorder=5)
        if labels:
            m = pts[len(pts) // 2] / s
            ax.annotate("SENW"[k], m, color=INK, fontsize=10, fontweight="bold", ha="center",
                        va="center", zorder=7,
                        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=c, lw=1.5))
    if corner_labels:
        for name, c, a in zip(["SW", "SE", "NE", "NW"], f["corners"], f["corner_angles"]):
            x, y = f["P"][c] / s
            ax.plot(x, y, "o", ms=8, mfc="white", mec=INK, mew=1.5, zorder=8)
            ax.annotate(f"{name} {a:.0f}°", (x, y), xytext=(6, 6), textcoords="offset points",
                        color=INK, fontsize=9, zorder=8,
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))


def split_tris(f, s=1e3, window=None):
    # Two Triangulations: Delaunay band and lattice squares (optionally only inside a window).
    nodes, tri, n_sq = f["nodes"] / s, f["tri"], f["n_sq"]
    parts = (tri[: len(tri) - n_sq], tri[len(tri) - n_sq:])
    out = []
    for t in parts:
        if window is not None:
            c = nodes[t].mean(axis=1)
            (x0, x1), (y0, y1) = window
            t = t[(c[:, 0] > x0) & (c[:, 0] < x1) & (c[:, 1] > y0) & (c[:, 1] < y1)]
        out.append(Triangulation(nodes[:, 0], nodes[:, 1], t))
    return out


def node_orthogonality(cg):
    # |90° - angle between grid lines| at each cell-corner node, as in tests/test_conformal.py::_ortho
    qx, qy = cg.x[::2, ::2], cg.y[::2, ::2]
    ax_, ay_, bx_, by_ = (np.gradient(q, axis=k) for k in (1, 0) for q in (qx, qy))
    cos = (ax_ * bx_ + ay_ * by_) / (np.hypot(ax_, ay_) * np.hypot(bx_, by_))
    return np.abs(90.0 - np.degrees(np.arccos(np.clip(cos, -1, 1))))

---
# (a) A trapezoid

1000 km wide at the bottom, 600 km at the top, 800 km tall, drawn as an exact trapezoid in a fixed
Lambert conformal plane (the same trick `tests/test_conformal.py` uses for its rectangle). The bottom
corners are 76°, the top ones 104°, so even this simple shape has two mildly singular corners.

Sanity check first: the test suite's 1000 × 600 km **rectangle** gives $E_\xi = 0.6$ exactly
and the interior rows of the stiffness matrix are the 5-point Laplacian (§5–7 of `03_dans_method.md`).

In [ ]:
PLANE = cf.MapProjection("lcc", -100.0, 35.0, 33.0, 37.0)

# the rectangle from tests/test_conformal.py
rlon, rlat = (np.asarray(a) for a in PLANE.to_lonlat([-5e5, 5e5, 5e5, -5e5], [-3e5, -3e5, 3e5, 3e5]))
rect = run_stages(rlon, rlat, [0, 1, 2, 3], 375, PLANE, 20_000, grid=False)
centre = np.argmin(np.hypot(*(rect["nodes"] - rect["nodes"].mean(0)).T))
row = rect["K"].getrow(centre)
print("stiffness row of the centre node:", np.round(np.sort(row.data), 12).tolist())

# the trapezoid
tlon, tlat = (np.asarray(a) for a in PLANE.to_lonlat([-5e5, 5e5, 3e5, -3e5], [-4e5, -4e5, 4e5, 4e5]))
trap_small = run_stages(tlon, tlat, [0, 1, 2, 3], 300, PLANE, 400)        # coarse, to see triangles
trap = run_stages(tlon, tlat, [0, 1, 2, 3], 300, PLANE, cf.BUILD_RASTER_POINTS)  # as Grid.from_outline builds

### Stage 1–2: the outline in the projection plane, sides and corners (§1–3)

`order_corners` makes the corner with the smallest $x+y$ the SW corner and walks
counter-clockwise; side $k$ runs from corner $k$ to corner $k+1$ (S, E, N, W). `_sides` puts
boundary nodes about $h$ apart along each side.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.2))
draw_outline(ax, trap_small)
for k, (pts, _) in enumerate(trap_small["sides"]):
    ax.plot(pts[:, 0] / 1e3, pts[:, 1] / 1e3, ".", color=SIDE["SENW"[k]], ms=4, zorder=6)
ax.set_aspect("equal"); ax.set_xlabel("x (km, Lambert conformal)"); ax.set_ylabel("y (km)")
ax.set_title(f"Trapezoid: sides, corners, and boundary nodes h = {trap_small['h']/1e3:.0f} km apart")
save(fig, "trapezoid_outline")

### Stage 3: the mesh (§4)

Blue: the structured lattice squares, each split into two right triangles. Orange: the scipy
Delaunay triangulation of the band of nodes within $3h$ of the outline, after dropping the
triangles whose centroid is in a square or outside the outline.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.2))
band, sq = split_tris(trap_small)
ax.triplot(sq, color=LATTICE, lw=0.6)
ax.triplot(band, color=BAND, lw=0.8)
draw_outline(ax, trap_small, labels=False, corner_labels=False)
ax.plot([], [], color=LATTICE, label=f"lattice squares ({trap_small['n_sq']} triangles)")
ax.plot([], [], color=BAND, label=f"Delaunay band ({len(trap_small['tri']) - trap_small['n_sq']} triangles)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)
ax.set_aspect("equal"); ax.set_xlabel("x (km)"); ax.set_ylabel("y (km)")
ax.set_title(f"Trapezoid mesh, raster_points=400 (h = {trap_small['h']/1e3:.0f} km)")
save(fig, "trapezoid_mesh")

### Stage 4: the two Laplace fields (§6)

$\xi$: 0 on W, 1 on E, zero normal flux on S and N. $\eta$: 0 on S, 1 on N, zero flux on E and W.
The contour lines drawn on top are the future grid lines.

In [ ]:
def plot_fields(f, name, title, n_lines=(10, 10), figsize=(10, 4.6)):
    tr = Triangulation(f["nodes"][:, 0] / 1e3, f["nodes"][:, 1] / 1e3, f["tri"])
    fig, axs = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)
    for ax, key, cmap, n in zip(axs, ("xi", "eta"), ("Blues", "Oranges"), n_lines):
        cs = ax.tricontourf(tr, f[key], levels=np.linspace(0, 1, 21), cmap=cmap)
        ax.tricontour(tr, f[key], levels=np.linspace(0, 1, n + 1)[1:-1], colors=INK, linewidths=0.4)
        draw_outline(ax, f, lw=1.6, corner_labels=False)
        ax.set_aspect("equal"); ax.set_xlabel("x (km)")
        sym = "ξ" if key == "xi" else "η"
        ax.set_title(f"{sym}  (E_{sym} = {f['E_' + key]:.4f})")
        fig.colorbar(cs, ax=ax, shrink=0.8, ticks=[0, 0.5, 1])
    axs[0].set_ylabel("y (km)")
    fig.suptitle(title, fontweight="bold")
    save(fig, name)


plot_fields(trap, "trapezoid_fields", f"Trapezoid: ξ and η on the build mesh ({len(trap['nodes']):,} nodes)")

### Stage 5: the mesh in image space (§7–8)

Each mesh node moved to $(\xi/M, \eta)$ with $M$ = `aspect`, so the image is the
$1/M \times 1$ rectangle the outline maps onto conformally. Small triangles keep their shape
(only scaled and rotated); that is what "conformal" means. Uniform cells in this rectangle are the
grid.

In [ ]:
def plot_image(f, name, title, figsize=(6, 5.2)):
    M = f["aspect"]
    nodes_img = np.column_stack([f["xi"] / M, f["eta"]])
    g = {"nodes": nodes_img * 1e3, "tri": f["tri"], "n_sq": f["n_sq"]}
    band, sq = split_tris(g)
    fig, ax = plt.subplots(figsize=figsize)
    ax.triplot(sq, color=LATTICE, lw=0.5)
    ax.triplot(band, color=BAND, lw=0.7)
    for k, (xy, lab) in enumerate([((0.5 / M, 0), "S"), ((1 / M, 0.5), "E"), ((0.5 / M, 1), "N"), ((0, 0.5), "W")]):
        ax.annotate(lab, xy, color=INK, fontweight="bold", ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=SIDE[lab], lw=1.5))
    ax.set_aspect("equal"); ax.set_xlabel("ξ / M"); ax.set_ylabel("η")
    ax.set_title(title)
    save(fig, name)


plot_image(trap_small, "trapezoid_image", f"Trapezoid mesh mapped to (ξ/M, η), M = {trap_small['aspect']:.4f}")

### Stage 6: the sampled supergrid (§8–9)

`_sample` interpolates $(x, y)$ at the uniform $(\xi, \eta)$ lattice, inside the image of each mesh
triangle; then `solve` snaps the outer rows/columns onto the sides and pins the 4 corners. Shown: the
cell-corner (q) lines of the supergrid.

In [ ]:
def plot_grid(f, name, title, figsize=(6, 5.2), cg=None):
    cg = cg or f["cg"]
    qx, qy = cg.x[::2, ::2] / 1e3, cg.y[::2, ::2] / 1e3
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(qx, qy, color=MUTED, lw=0.5)
    ax.plot(qx.T, qy.T, color=MUTED, lw=0.5)
    draw_outline(ax, f, lw=1.8)
    ax.set_aspect("equal"); ax.set_xlabel("x (km)"); ax.set_ylabel("y (km)")
    ax.set_title(title)
    save(fig, name)


plot_grid(trap, "trapezoid_grid", f"Trapezoid grid: nx × ny = {trap['nx']} × {trap['ny']}")

### Stage 7: what the grid looks like: cell size and orthogonality (§10)

Left: true cell size $\sqrt{dx\,dy}$ (km) from `_cell_geometry` (projected lengths divided by the
projection's scale factor). Right: deviation from 90° between grid lines at each cell-corner node,
on a log scale. The obtuse top corners make the smallest cells; the error is largest at the four
corner nodes, where a 76° or 104° outline corner can't be 90°.

In [ ]:
def plot_quality(f, name, title, figsize=(10, 4.6), cg=None):
    cg = cg or f["cg"]
    qx, qy, size, _ = cf._cell_geometry(cg)
    ortho = node_orthogonality(cg)
    fig, axs = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)
    pc = axs[0].pcolormesh(qx / 1e3, qy / 1e3, size, cmap="Greens", shading="flat",
                           norm=LogNorm(size.min(), size.max()))
    fig.colorbar(pc, ax=axs[0], shrink=0.8, label="true cell size (km, log scale)")
    pc = axs[1].pcolormesh(qx / 1e3, qy / 1e3, np.maximum(ortho, 1e-3), shading="gouraud",
                           cmap="Purples", norm=LogNorm(1e-3, 90))
    fig.colorbar(pc, ax=axs[1], shrink=0.8, label="|90° − grid angle| (degrees)")
    p1, p50, p99 = np.percentile(size, [1, 50, 99])
    axs[0].set_title(f"true cell size\np1 {p1:.1f} · median {p50:.1f} · p99 {p99:.1f} km")
    inner = ortho[2:-2, 2:-2]
    axs[1].set_title(f"|90° − grid angle| at cell corners\nmedian {np.median(ortho):.2f}° · "
                     f"p99 {np.percentile(ortho, 99):.1f}° · max {ortho.max():.0f}°")
    for ax in axs:
        draw_outline(ax, f, lw=1.2, labels=False, corner_labels=False)
        ax.set_aspect("equal"); ax.set_xlabel("x (km)")
    axs[0].set_ylabel("y (km)")
    fig.suptitle(title, fontweight="bold")
    save(fig, name)
    print(f"interior (skip 2 node rings): ortho p99={np.percentile(inner, 99):.2f}°, max={inner.max():.1f}°")


plot_quality(trap, "trapezoid_quality", "Trapezoid, build mesh (250k points)")

---
# (b) California

The 30-vertex outline in `tests/data/california_bays.outline.json`, corners `[0, 6, 22, 29]`,
12,000 cells. `projection=None` gives `projection_for("auto")`, which is Lambert conformal here,
as in the tests. (`Grid.from_outline(projection="auto")` would pick Mercator instead, see the
reviewer cell at the end.)

The logical sides are not compass directions here: "W" is the offshore line, "E" is the coast
from Los Angeles to the East Bay, and the grid's long axis ($\eta$, 193 cells) runs along the coast.

In [ ]:
WT = WORKTREE
cal = json.loads((WT / "tests/data/california_bays.outline.json").read_text())["outline"]
CLON, CLAT, CCORN = cal["lon"], cal["lat"], cal["corners"]

cal_small = run_stages(CLON, CLAT, CCORN, 12_000, None, 2_000)     # to see triangles
cal_mid = run_stages(CLON, CLAT, CCORN, 12_000, None, 20_000, grid=False)  # fields (preview size)
cal = run_stages(CLON, CLAT, CCORN, 12_000, None, cf.BUILD_RASTER_POINTS)  # build size

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 7))
draw_outline(ax, cal)
P = cal["P"] / 1e3
ax.plot(P[:, 0], P[:, 1], "o", ms=3, color=INK, zorder=6)
for i, (x, y) in enumerate(P):
    if i not in cal["corners"]:
        ax.annotate(str(i), (x, y), xytext=(3, -8), textcoords="offset points", fontsize=7, color=MUTED)
ax.set_aspect("equal"); ax.set_xlabel("x (km, Lambert conformal)"); ax.set_ylabel("y (km)")
ax.set_title("California outline: 30 vertices, 4 corners")
save(fig, "california_outline")

### The mesh, and a zoom on the 204° corner (§4)

Left: the whole mesh at `raster_points=2000` ($h \approx 12$ km) so the triangles are visible.
Right: the build mesh ($h \approx 1.09$ km) round the NE corner (vertex 22, 203.5°, in the East
Bay). The dashed contours are the distance to the outline in units of $h$ at the three thresholds
`_mesh` uses: lattice nodes are kept beyond 0.5, squares need all four corners beyond 1.5, and
lattice nodes enter the Delaunay band inside 3.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6.4), gridspec_kw=dict(width_ratios=[1, 1.15]))
band, sq = split_tris(cal_small)
axs[0].triplot(sq, color=LATTICE, lw=0.4)
axs[0].triplot(band, color=BAND, lw=0.5)
draw_outline(axs[0], cal_small, lw=1.5, labels=False)
axs[0].set_title(f"raster_points=2,000, h = {cal_small['h']/1e3:.1f} km")

# zoom on the NE corner of the build mesh
cx, cy = cal["P"][cal["corners"][2]] / 1e3
r = 9.0
win = ((cx - r, cx + r), (cy - r, cy + r))
band, sq = split_tris(cal, window=((cx - 1.5 * r, cx + 1.5 * r), (cy - 1.5 * r, cy + 1.5 * r)))
axs[1].triplot(sq, color=LATTICE, lw=0.6)
axs[1].triplot(band, color=BAND, lw=0.8)
draw_outline(axs[1], cal, lw=2.0, labels=False, corner_labels=False)
cs = axs[1].contour(cal["X"] / 1e3, cal["Y"] / 1e3, cal["d"], levels=[0.5, 1.5, 3.0],
                    colors=INK, linewidths=1.0, linestyles="--")
axs[1].clabel(cs, fmt=lambda v: f"d = {v:g} h", fontsize=8)
axs[1].plot(cx, cy, "o", ms=9, mfc="white", mec=INK, mew=1.5, zorder=8)
axs[1].annotate(f"NE corner, {cal['corner_angles'][2]:.1f}°", (cx, cy), xytext=(8, 8),
                textcoords="offset points", fontsize=9, zorder=9,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))
axs[1].set_xlim(*win[0]); axs[1].set_ylim(*win[1])
axs[1].text(0.02, 0.02, "dashed: distance from the outline\n= 0.5 h, 1.5 h, 3 h (outline inwards)",
            transform=axs[1].transAxes, fontsize=9, color=INK, va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=MUTED))
axs[1].set_title(f"zoom on the build mesh, h = {cal['h']/1e3:.2f} km")
for ax in axs:
    ax.set_aspect("equal"); ax.set_xlabel("x (km)")
axs[0].set_ylabel("y (km)")
axs[0].plot([], [], color=LATTICE, label="lattice squares")
axs[0].plot([], [], color=BAND, label="Delaunay band")
axs[0].legend(loc="lower left", frameon=True)
save(fig, "california_mesh")

In [ ]:
plot_fields(cal_mid, "california_fields",
            f"California: ξ (across-shore) and η (along-shore), {len(cal_mid['nodes']):,}-node mesh",
            n_lines=(8, 24), figsize=(11, 6.8))

In [ ]:
plot_image(cal_small, "california_image",
           f"California mesh (2,000 points) in (ξ/M, η), M = {cal_small['aspect']:.4f}", figsize=(5, 8))

The grid figure uses 3,000 cells instead of 12,000 so the lines stay visible.

In [ ]:
cal_cg_3k = cf.solve(CLON, CLAT, CCORN, 3_000, None)
plot_grid(cal, "california_grid", f"California grid, 3,000 cells: {cal_cg_3k.nx} × {cal_cg_3k.ny}",
          figsize=(6.5, 7), cg=cal_cg_3k)

In [ ]:
plot_quality(cal, "california_quality",
             f"California, 12,000 cells ({cal['nx']} × {cal['ny']}), build mesh", figsize=(11, 6.8))

### Convergence of the two energies (§7)

$E_{\xi,h}$ is an upper bound on the modulus $M$ and $1/E_{\eta,h}$ a lower bound (both FE energies
are upper bounds of the exact ones, and $E_\xi E_\eta = 1$ exactly). `aspect` is their geometric
mean. The bracket closes slowly because of the corner singularities (§11).

In [ ]:
rps = [1_000, 4_000, 20_000, 100_000, 250_000]
up, lowb, asp = [], [], []
for rp in rps:
    f = cf._solve_mesh(CLON, CLAT, CCORN, 12_000, None, rp)
    K = cf._stiffness(f["nodes"], f["tri"])
    e_xi = cf._harmonic(K, f["ids"][3], f["ids"][1])[1]
    e_eta = cf._harmonic(K, f["ids"][0], f["ids"][2])[1]
    up.append(e_xi); lowb.append(1 / e_eta); asp.append(f["aspect"])
    print(f"raster_points={rp:>7,}: E_xi={e_xi:.4f}  1/E_eta={1/e_eta:.4f}  aspect={f['aspect']:.4f}  "
          f"E_xi*E_eta-1={e_xi*e_eta-1:.2e}")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(rps, up, "-o", color=SIDE["S"], lw=2, ms=8, label="E_ξ  (upper bound)")
ax.plot(rps, lowb, "-o", color=SIDE["E"], lw=2, ms=8, label="1 / E_η  (lower bound)")
ax.plot(rps, asp, "--", color=MUTED, lw=1.5, label="aspect = √(E_ξ/E_η)")
ax.set_xscale("log"); ax.set_xlabel("raster_points (≈ mesh nodes)"); ax.set_ylabel("conformal modulus M")
ax.annotate("E_ξ", (rps[0], up[0]), xytext=(8, 0), textcoords="offset points", va="center", color=INK)
ax.annotate("1/E_η", (rps[0], lowb[0]), xytext=(8, 0), textcoords="offset points", va="center", color=INK)
ax.legend(frameon=False)
ax.grid(color="#e6e5e0", lw=0.6)
ax.set_title("California: the FE energies bracket the modulus")
save(fig, "california_energy")

---
# Reviewer checks (§13 of `03_dans_method.md`)

### Two "auto" choosers, and what the projection actually changes

`cf.solve(projection=None)` uses `projection_for("auto")` (a latitude rule); `Grid.from_outline`
uses the smallest `size_ratios`. They disagree on California. The aspect then differs because a
straight edge between two vertices is a different curve on the Earth in each plane; densifying the
outline along geodesics makes the four projections agree (see `01_the_problem.md` §1.7 and
`projection_invariance.ipynb` for the full version of this check).

In [ ]:
from pyproj import Geod

print("projection_for('auto'):", cf.projection_for("auto", CLON, CLAT).kind)
r = cf.size_ratios(CLON, CLAT, CCORN)
print("size_ratios:", {k: round(v, 3) for k, v in r.items()}, "-> Grid.from_outline picks", min(r, key=r.get))

geod = Geod(ellps="WGS84")


def densify(lon, lat, corners, km=5.0):
    L, A, at = [], [], {}
    for i in range(len(lon)):
        j = (i + 1) % len(lon)
        at[i] = len(L)
        L.append(lon[i]); A.append(lat[i])
        k = int(geod.inv(lon[i], lat[i], lon[j], lat[j])[2] / 1e3 // km)
        for p in (geod.npts(lon[i], lat[i], lon[j], lat[j], k) if k else []):
            L.append(p[0]); A.append(p[1])
    return L, A, [at[c] for c in corners]


dl, da, dc = densify(CLON, CLAT, CCORN)
for kind in cf.KINDS:
    a = cf._solve_mesh(CLON, CLAT, CCORN, 12_000, kind, 20_000)
    b = cf._solve_mesh(dl, da, dc, 12_000, kind, 20_000)
    print(f"{kind:5s}: aspect, edges straight in the plane {a['aspect']:.4f} ({a['nx']}x{a['ny']});"
          f"  geodesic edges {b['aspect']:.4f} ({b['nx']}x{b['ny']})")

### Does `_sample`'s "last write wins" or the nearest-node fallback ever matter?

Count image triangles whose orientation flips between physical and $(\xi,\eta)$ space (overlapping
images, where the last triangle written wins), and lattice points that land in no image triangle (where
`solve` falls back to the nearest node). On California and the trapezoid both are zero.

In [ ]:
def image_checks(f):
    nodes, tri = f["nodes"], f["tri"]
    uv = np.column_stack([f["xi"], f["eta"]])

    def signed2(p):
        a = p[tri]
        return (a[:, 1, 0] - a[:, 0, 0]) * (a[:, 2, 1] - a[:, 0, 1]) - (a[:, 1, 1] - a[:, 0, 1]) * (a[:, 2, 0] - a[:, 0, 0])

    s_phys, s_img = signed2(nodes), signed2(uv)
    flipped = int(((np.sign(s_phys) != np.sign(s_img)) & (s_img != 0)).sum())
    XY = cf._sample(uv, tri, nodes, 0.0, 0.5 / np.array([f["nx"], f["ny"]]), (2 * f["ny"] + 1, 2 * f["nx"] + 1))
    misses = int(np.isnan(XY[..., 0]).sum())
    return flipped, int((s_img == 0).sum()), misses


for name, f in [("trapezoid 400", trap_small), ("trapezoid 250k", trap), ("California 2k", cal_small),
                ("California 20k", cal_mid), ("California 250k", cal)]:
    fl, deg, miss = image_checks(f)
    print(f"{name:16s}: flipped image triangles={fl}, zero-area image triangles={deg}, "
          f"lattice points in no image triangle (fallback)={miss}")

### A thin, bent spit: the Delaunay band is not a constrained triangulation

A 100 km box with a 0.2 km-wide zig-zag spit of land cut into it from the south. At
`raster_points=1000` ($h \approx 3.2$ km) some band triangles straddle the spit: their centroids are
inside the outline, so `_mesh` keeps them, and the fields leak across the land. Red: kept triangles
that are partly outside the outline. Right: the grid from that mesh near the spit, folded cells
filled red.

In [ ]:
def zigzag_box(w=0.2, n=4, amp=8.0, half=50.0):
    a = w / 2
    ys = np.linspace(-half, 0, n + 1)
    xs = np.array([0.0] + [amp if k % 2 else -amp for k in range(1, n + 1)])
    spit = [(x - a, y) for x, y in zip(xs, ys)] + [(x + a, y) for x, y in zip(xs, ys)][::-1]
    pts = [(-half, -half)] + spit + [(half, -half), (half, half), (-half, half)]
    x, y = (np.array(v) * 1e3 for v in zip(*pts))
    return x, y, [0, len(pts) - 3, len(pts) - 2, len(pts) - 1]


zx, zy, zc = zigzag_box()
zlon, zlat = (np.asarray(v) for v in PLANE.to_lonlat(zx, zy))
poly = Polygon(np.column_stack([zx, zy]))
for rp in (1_000, 20_000, 250_000):
    f = cf._solve_mesh(zlon, zlat, zc, 2_500, PLANE, rp)
    tris = shapely.polygons(f["nodes"][f["tri"]])
    out = shapely.area(tris) - shapely.area(shapely.intersection(tris, poly))
    print(f"raster_points={rp:>7,} (h={f['h']/1e3:.2f} km): kept triangles partly outside={int((out > 1e-6 * shapely.area(tris)).sum())}, "
          f"area outside={out.sum()/1e6:.2f} km², aspect={f['aspect']:.4f}")

zf = cf._solve_mesh(zlon, zlat, zc, 2_500, PLANE, 1_000)
zcg = cf.solve(zlon, zlat, zc, 2_500, PLANE, 1_000)
tris = shapely.polygons(zf["nodes"][zf["tri"]])
out = shapely.area(tris) - shapely.area(shapely.intersection(tris, poly))
bridging = out > 1e-6 * shapely.area(tris)

qx, qy = zcg.x[::2, ::2] / 1e3, zcg.y[::2, ::2] / 1e3
xs = [qx[:-1, :-1], qx[:-1, 1:], qx[1:, 1:], qx[1:, :-1]]
ys = [qy[:-1, :-1], qy[:-1, 1:], qy[1:, 1:], qy[1:, :-1]]
area = sum(xs[i] * ys[(i + 1) % 4] - xs[(i + 1) % 4] * ys[i] for i in range(4)) / 2
print("folded cells (signed area <= 0) in the rp=1000 grid:", int((area <= 0).sum()))

ref = cf.solve(zlon, zlat, zc, 2_500, PLANE, 250_000)


def draw_q(ax, cg, folded=None):
    qx, qy = cg.x[::2, ::2] / 1e3, cg.y[::2, ::2] / 1e3
    ax.plot(qx, qy, color=MUTED, lw=0.5); ax.plot(qx.T, qy.T, color=MUTED, lw=0.5)
    if folded is not None:
        for j, i in zip(*np.nonzero(folded)):
            ax.fill([qx[j, i], qx[j, i + 1], qx[j + 1, i + 1], qx[j + 1, i]],
                    [qy[j, i], qy[j, i + 1], qy[j + 1, i + 1], qy[j + 1, i]], color=ALERT, alpha=0.7)


fig, axs = plt.subplots(1, 3, figsize=(15, 6.2), gridspec_kw=dict(width_ratios=[1.3, 1, 1], wspace=0.3))
tr = Triangulation(zf["nodes"][:, 0] / 1e3, zf["nodes"][:, 1] / 1e3, zf["tri"])
axs[0].triplot(tr, color="#c9c8c2", lw=0.4)
for t in zf["tri"][bridging]:
    p = zf["nodes"][t] / 1e3
    axs[0].fill(p[:, 0], p[:, 1], color=ALERT, alpha=0.6, lw=0)
axs[0].set_title(f"mesh, raster_points=1,000 (h={zf['h']/1e3:.1f} km):\n{int(bridging.sum())} kept triangles cross the 0.2 km spit")
draw_q(axs[1], zcg, folded=area <= 0)
axs[1].set_title(f"grid from that mesh: {zcg.nx} × {zcg.ny},\naspect {zcg.aspect:.3f}, {int((area <= 0).sum())} folded cells (red)")
draw_q(axs[2], ref)
axs[2].set_title(f"build mesh (h={ref.h/1e3:.1f} km): {ref.nx} × {ref.ny},\naspect {ref.aspect:.3f}, no bridging")
for ax in axs:
    ax.plot(np.r_[zx, zx[:1]] / 1e3, np.r_[zy, zy[:1]] / 1e3, color=INK, lw=1.2)
    ax.set_aspect("equal"); ax.set_xlabel("x (km)")
for ax in axs[1:]:
    ax.set_xlim(-20, 20); ax.set_ylim(-52, 8)
axs[0].set_ylabel("y (km)")
save(fig, "bridging")